# Model Building (Updated for `light_curves.csv`)
**Purpose:** Train a self-supervised contrastive learning encoder (SimCLR-style) on multi-band light-curve sequences.

**Inputs (preferred):**
- `outputs/light_curve_pairs.npz` from `Data_Augmentation_updated_for_light_curves.ipynb`
- or `outputs/light_curves_sequences.npz` from `Preprocessing_updated_for_light_curves.ipynb`

**Outputs:**
- Trained encoder + projection head checkpoints
- Validation embeddings + clustering metrics (silhouette)


In [1]:
import os
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler


# =========================
# 1) Config
# =========================
CSV_PATH = "outputs/selected_features_full.csv"   # change if needed
OUT_PKL  = "simclr_model.pkl"

BATCH_SIZE = 512
EPOCHS = 30
LR = 1e-3
TEMPERATURE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Pick the columns you want to train on (simple + explicit)
FEATURE_COLS = ["mjd", "fid", "magpsf", "sigmapsf", "ra", "dec", "isdiffpos"]


# =========================
# 2) Dataset + augmentations
# =========================
class SimCLRTabularDataset(Dataset):
    def __init__(self, X: np.ndarray, noise_std: float = 0.02, drop_prob: float = 0.10):
        """
        X: standardized numpy array (N, D)
        noise_std: gaussian noise strength (on standardized features)
        drop_prob: probability of masking a feature to 0 (feature dropout)
        """
        self.X = X.astype(np.float32)
        self.noise_std = noise_std
        self.drop_prob = drop_prob

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        # 1) Gaussian noise
        x = x + torch.randn_like(x) * self.noise_std

        # 2) Feature dropout (randomly mask some features)
        if self.drop_prob > 0:
            mask = (torch.rand_like(x) > self.drop_prob).float()
            x = x * mask

        return x

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        x1 = self._augment(x.clone())
        x2 = self._augment(x.clone())
        return x1, x2


# =========================
# 3) SimCLR Model (MLP encoder + projection head)
# =========================
class MLPEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, x):
        return self.net(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)


class SimCLR(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, emb_dim: int = 128, proj_dim: int = 128):
        super().__init__()
        self.encoder = MLPEncoder(in_dim, hidden_dim=hidden_dim, emb_dim=emb_dim)
        self.projector = ProjectionHead(emb_dim, proj_dim=proj_dim)

    def forward(self, x):
        h = self.encoder(x)              # (B, emb_dim)
        z = self.projector(h)            # (B, proj_dim)
        z = F.normalize(z, dim=1)
        return h, z


# =========================
# 4) NT-Xent loss
# =========================
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.2) -> torch.Tensor:
    """
    z1, z2: normalized (B, D)
    """
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # (2B, D)

    # cosine similarity matrix
    sim = torch.mm(z, z.t()) / temperature  # (2B, 2B)

    # mask self-similarity
    mask = torch.eye(2 * B, device=z.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # positives: (i, i+B) and (i+B, i)
    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], dim=0)  # (2B,)

    # denominator: logsumexp over all except itself
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()


# =========================
# 5) Train
# =========================
def main():
    # Load
    df = pd.read_csv(CSV_PATH)

    # Keep only required feature columns that exist
    cols = [c for c in FEATURE_COLS if c in df.columns]
    if len(cols) < 2:
        raise ValueError(f"Not enough usable feature columns found. Found: {cols}")

    Xdf = df[cols].copy()

    # Convert to numeric + clean
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")
    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)

    # Fill NaNs with median
    Xdf = Xdf.fillna(Xdf.median(numeric_only=True))

    # Standardize
    scaler = StandardScaler()
    X = scaler.fit_transform(Xdf.values)

    # DataLoader
    ds = SimCLRTabularDataset(X, noise_std=0.03, drop_prob=0.10)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)

    # Model
    model = SimCLR(in_dim=X.shape[1], hidden_dim=256, emb_dim=128, proj_dim=128).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    model.train()
    for epoch in range(1, EPOCHS + 1):
        losses = []
        for x1, x2 in dl:
            x1 = x1.to(DEVICE)
            x2 = x2.to(DEVICE)

            _, z1 = model(x1)
            _, z2 = model(x2)

            loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)

            opt.zero_grad()
            loss.backward()
            opt.step()

            losses.append(loss.item())

        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={np.mean(losses):.4f}")

    # =========================
    # 6) Save as PKL
    # =========================
    package = {
        "feature_cols": cols,
        "scaler": scaler,
        "model_state_dict": model.state_dict(),
        "model_config": {
            "in_dim": X.shape[1],
            "hidden_dim": 256,
            "emb_dim": 128,
            "proj_dim": 128
        }
    }

    with open(OUT_PKL, "wb") as f:
        pickle.dump(package, f)

    print(f"\n✅ Saved SimCLR package to: {OUT_PKL}")


if __name__ == "__main__":
    main()


Epoch 01/30 | loss=4.6669
Epoch 02/30 | loss=4.0965
Epoch 03/30 | loss=3.8928
Epoch 04/30 | loss=3.7769
Epoch 05/30 | loss=3.6611
Epoch 06/30 | loss=3.5972
Epoch 07/30 | loss=3.5359
Epoch 08/30 | loss=3.4849
Epoch 09/30 | loss=3.4653
Epoch 10/30 | loss=3.4448
Epoch 11/30 | loss=3.4220
Epoch 12/30 | loss=3.3943
Epoch 13/30 | loss=3.3886
Epoch 14/30 | loss=3.3654
Epoch 15/30 | loss=3.3646
Epoch 16/30 | loss=3.3436
Epoch 17/30 | loss=3.3210
Epoch 18/30 | loss=3.3168
Epoch 19/30 | loss=3.3161
Epoch 20/30 | loss=3.3005
Epoch 21/30 | loss=3.2945
Epoch 22/30 | loss=3.2823
Epoch 23/30 | loss=3.2888
Epoch 24/30 | loss=3.2831
Epoch 25/30 | loss=3.2694
Epoch 26/30 | loss=3.2520
Epoch 27/30 | loss=3.2565
Epoch 28/30 | loss=3.2558
Epoch 29/30 | loss=3.2504
Epoch 30/30 | loss=3.2424

✅ Saved SimCLR package to: simclr_model.pkl


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pickle

# =========================
# CONFIG
# =========================
CSV_PATH = "labeled_dataset.csv"
FEATURE_COLS = ["mjd", "fid", "magpsf", "sigmapsf", "ra", "dec", "isdiffpos"]
LABEL_COL = "transient_type"

BATCH_SIZE = 256
EPOCHS = 30
LR = 1e-3
TEMPERATURE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(CSV_PATH)

X = df[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

y = df[LABEL_COL].astype(str)

# =========================
# STANDARDIZE
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================
# SIMCLR DATASET
# =========================
class SimCLRDataset(Dataset):
    def __init__(self, X, noise=0.03, drop=0.1):
        self.X = X.astype(np.float32)
        self.noise = noise
        self.drop = drop

    def augment(self, x):
        x = x + torch.randn_like(x) * self.noise
        mask = (torch.rand_like(x) > self.drop).float()
        return x * mask

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx])
        return self.augment(x), self.augment(x)


# =========================
# SIMCLR MODEL
# =========================
class Encoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.net(x)


class SimCLR(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.encoder = Encoder(d)
        self.projector = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, F.normalize(z, dim=1)


def nt_xent(z1, z2, temp=0.2):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.T) / temp
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([torch.diag(sim, N), torch.diag(sim, -N)])
    loss = -positives + torch.logsumexp(sim, dim=1)
    return loss.mean()


# =========================
# TRAIN SIMCLR
# =========================
dataset = SimCLRDataset(X_scaled)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = SimCLR(X.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    losses = []
    for x1, x2 in loader:
        x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
        _, z1 = model(x1)
        _, z2 = model(x2)

        loss = nt_xent(z1, z2)
        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {np.mean(losses):.4f}")

# =========================
# EMBEDDING EXTRACTION
# =========================
with torch.no_grad():
    H, _ = model(torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE))
    H = H.cpu().numpy()

# =========================
# CLASSIFIER
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    H, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

print("\nClassifier accuracy:", clf.score(X_test, y_test))

# =========================
# SAVE EVERYTHING
# =========================
with open("simclr_classifier_pipeline.pkl", "wb") as f:
    pickle.dump({
        "scaler": scaler,
        "encoder_state": model.state_dict(),
        "feature_cols": FEATURE_COLS,
        "classifier": clf
    }, f)

print("✅ Saved: simclr_classifier_pipeline.pkl")


Epoch 1/30 | Loss: 3.7385
Epoch 2/30 | Loss: 2.9760
Epoch 3/30 | Loss: 2.8422
Epoch 4/30 | Loss: 2.6243
Epoch 5/30 | Loss: 2.3780
Epoch 6/30 | Loss: 2.3081
Epoch 7/30 | Loss: 2.2130
Epoch 8/30 | Loss: 2.1249
Epoch 9/30 | Loss: 2.0611
Epoch 10/30 | Loss: 2.1660
Epoch 11/30 | Loss: 1.9799
Epoch 12/30 | Loss: 2.0640
Epoch 13/30 | Loss: 1.9647
Epoch 14/30 | Loss: 1.9286
Epoch 15/30 | Loss: 2.1230
Epoch 16/30 | Loss: 2.0395
Epoch 17/30 | Loss: 1.9801
Epoch 18/30 | Loss: 2.0332
Epoch 19/30 | Loss: 2.1156
Epoch 20/30 | Loss: 2.0371
Epoch 21/30 | Loss: 1.8886
Epoch 22/30 | Loss: 1.8207
Epoch 23/30 | Loss: 1.9505
Epoch 24/30 | Loss: 1.9203
Epoch 25/30 | Loss: 2.0229
Epoch 26/30 | Loss: 2.0694
Epoch 27/30 | Loss: 1.7965
Epoch 28/30 | Loss: 1.9248
Epoch 29/30 | Loss: 1.9619
Epoch 30/30 | Loss: 1.7879

Classifier accuracy: 0.7272727272727273
✅ Saved: simclr_classifier_pipeline.pkl


In [5]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- same SimCLR structure you trained with ---
class Encoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
    def forward(self, x):
        return self.net(x)

class SimCLR(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.encoder = Encoder(d)
        self.projector = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, F.normalize(z, dim=1)


def predict_sample(sample_dict):
    # Load pipeline
    with open("simclr_classifier_pipeline.pkl", "rb") as f:
        pkg = pickle.load(f)

    feature_cols = pkg["feature_cols"]
    scaler = pkg["scaler"]
    clf = pkg["classifier"]

    # ✅ Load FULL SimCLR (because state_dict contains encoder+projector keys)
    simclr = SimCLR(len(feature_cols))
    simclr.load_state_dict(pkg["encoder_state"])   # (this is actually full simclr state dict)
    simclr.eval()

    # Build input in correct order
    x = np.array([[float(sample_dict[c]) for c in feature_cols]], dtype=np.float32)

    # Scale
    x_scaled = scaler.transform(x)

    # Get embedding
    with torch.no_grad():
        xt = torch.tensor(x_scaled, dtype=torch.float32)
        h, _ = simclr(xt)
        h_np = h.numpy()

    # Predict
    pred = clf.predict(h_np)[0]
    return pred


# Example
sample = {
    "mjd": 58278.40,
    "fid": 1,
    "magpsf": 16.69,
    "sigmapsf": 0.025,
    "ra": 307.79,
    "dec": 51.13,
    "isdiffpos": -1
}

print("Predicted class:", predict_sample(sample))


Predicted class: Supernova_Ia


c:\Users\NIPUN\.conda\envs\torch_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [1]:
# If needed (Colab / fresh env), uncomment:
# !pip -q install tensorflow numpy pandas scikit-learn umap-learn matplotlib


In [2]:
import os, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)


c:\Users\NIPUN\.conda\envs\torch_env\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TF: 2.20.0


In [3]:
# ---- Build sequences NPZ from selected features CSV if present ----
import os, glob
import numpy as np
import pandas as pd

SELECTED_CSV = os.path.join('outputs', 'selected_features_full.csv')
OUT_SEQ = os.path.join('outputs', 'light_curves_sequences_from_selected.npz')

def build_seqs_from_selected(selected_csv, raw_csv='light_curves.csv', out_npz=OUT_SEQ):
    if not os.path.exists(selected_csv):
        return None
    # read header of selected to know which columns to keep
    sel_cols = pd.read_csv(selected_csv, nrows=0).columns.tolist()
    # load raw full CSV to get oid and mjd
    df_all = pd.read_csv(raw_csv)
    if 'oid' not in df_all.columns or 'mjd' not in df_all.columns:
        print('raw CSV missing oid/mjd; cannot build sequences')
        return None
    # choose available columns: preserve oid,mjd and selected features present in raw
    features = [c for c in sel_cols if c in df_all.columns and c not in ['mjd','fid'] ]
    # always include fid as numeric feature if present
    if 'fid' in sel_cols and 'fid' in df_all.columns and 'fid' not in features:
        features.append('fid')
    # ensure features is not empty
    if len(features) == 0:
        print('No matching feature columns found in raw CSV')
        return None
    cols = ['oid','mjd'] + features
    df = df_all[cols].dropna(subset=['oid','mjd']).sort_values(['oid','mjd']).reset_index(drop=True)
    groups = list(df.groupby('oid'))
    N = len(groups)
    T = max(len(g[1]) for g in groups)
    F = len(features)
    X = np.zeros((N, T, F), dtype=np.float32)
    oids = []
    for i, (oid, g) in enumerate(groups):
        oids.append(oid)
        vals = g[features].to_numpy(dtype=np.float32)
        L = vals.shape[0]
        X[i, :L, :] = vals
    np.savez_compressed(out_npz, X=X, oids=np.array(oids, dtype=object), feature_names=np.array(features, dtype=object))
    print('Wrote sequence NPZ from selected CSV:', out_npz, 'shape:', X.shape, 'features:', features)
    return out_npz

# build if needed and prefer the generated NPZ
generated = build_seqs_from_selected(SELECTED_CSV)
if generated:
    print('Generated sequences NPZ to be used for training:', generated)
else:
    print('No selected CSV found or could not build sequences from it.')

Wrote sequence NPZ from selected CSV: outputs\light_curves_sequences_from_selected.npz shape: (5, 4310, 6) features: ['magpsf', 'sigmapsf', 'ra', 'dec', 'isdiffpos', 'fid']
Generated sequences NPZ to be used for training: outputs\light_curves_sequences_from_selected.npz


In [4]:
# -----------------------
# Paths (update if you keep a different folder structure)
# -----------------------
# Prefer the precomputed augmented pairs (two views per object)
PAIRS_NPZ_CANDIDATES = [
    "outputs/light_curve_pairs.npz",
    "augmented/light_curve_pairs.npz",
    "augmented/ztf_obscontext_pairs.npz",  # legacy name (won't exist in new pipeline)
]

# Fallback: fixed-length sequences per object (we will augment on-the-fly)
SEQS_NPZ_CANDIDATES = [
    "outputs/light_curves_sequences.npz",
    "outputs/light_curves_sequences_selected.npz",
    "outputs/light_curves_sequences_fixed.npz",
]

CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def pick_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

pairs_path = pick_existing(PAIRS_NPZ_CANDIDATES)
seqs_path  = pick_existing(SEQS_NPZ_CANDIDATES)

print("pairs_path:", pairs_path)
print("seqs_path :", seqs_path)

if pairs_path is None and seqs_path is None:
    raise FileNotFoundError(
        f"""Couldn't find inputs. Run preprocessing + augmentation first, or put NPZ files in /outputs.
Looked for pairs: {PAIRS_NPZ_CANDIDATES}
Looked for seqs : {SEQS_NPZ_CANDIDATES}
"""
    )


pairs_path: outputs/light_curve_pairs.npz
seqs_path : None


In [5]:
# -----------------------
# Load data
# -----------------------
def load_pairs_npz(path: str):
    d = np.load(path, allow_pickle=True)
    # Expected keys: X1, X2, oid, feature_names
    X1 = d["X1"].astype(np.float32)
    X2 = d["X2"].astype(np.float32)
    oid = d["oid"] if "oid" in d else None
    feature_names = d["feature_names"].tolist() if "feature_names" in d else None
    return X1, X2, oid, feature_names

def load_seqs_npz(path: str):
    d = np.load(path, allow_pickle=True)
    # Expected keys: X, oid, feature_names
    X = d["X"].astype(np.float32)
    oid = d["oid"] if "oid" in d else None
    feature_names = d["feature_names"].tolist() if "feature_names" in d else None
    return X, oid, feature_names

if pairs_path:
    X1, X2, oid, feature_names = load_pairs_npz(pairs_path)
    X = None
    print("Loaded PAIRS:", X1.shape, X2.shape, "features:", len(feature_names) if feature_names else None)
else:
    X, oid, feature_names = load_seqs_npz(seqs_path)
    print("Loaded SEQS:", X.shape, "features:", len(feature_names) if feature_names else None)

T = (X1.shape[1] if pairs_path else X.shape[1])
F = (X1.shape[2] if pairs_path else X.shape[2])
print("Sequence length T:", T, "Num features F:", F)


Loaded PAIRS: (5, 128, 3) (5, 128, 3) features: 3
Sequence length T: 128 Num features F: 3


In [6]:
# -----------------------
# If only sequences are available: define on-the-fly augmentation to create two views
# -----------------------
def augment_view(x, noise_std=0.02, mag_shift=0.02, time_mask_p=0.1):
    """x: (T, F)"""
    x = tf.identity(x)

    # Gaussian noise
    x = x + tf.random.normal(tf.shape(x), stddev=noise_std, dtype=x.dtype)

    # Magnitude shift / scaling (apply to all features, small)
    scale = 1.0 + tf.random.uniform([], -mag_shift, mag_shift, dtype=x.dtype)
    shift = tf.random.uniform([], -mag_shift, mag_shift, dtype=x.dtype)
    x = x * scale + shift

    # Random time masking: zero-out some timesteps
    if time_mask_p and time_mask_p > 0:
        mask = tf.cast(tf.random.uniform([tf.shape(x)[0], 1], 0, 1) > time_mask_p, x.dtype)
        x = x * mask

    return x

def make_pair_from_seq(x):
    v1 = augment_view(x)
    v2 = augment_view(x)
    return v1, v2


In [7]:
# -----------------------
# Train/Val split (by object)
# -----------------------
if pairs_path:
    idx = np.arange(X1.shape[0])
else:
    idx = np.arange(X.shape[0])

train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=SEED, shuffle=True)

print("train:", len(train_idx), "val:", len(val_idx))


train: 4 val: 1


In [8]:
# -----------------------
# Build tf.data datasets
# -----------------------
BATCH = 256
AUTOTUNE = tf.data.AUTOTUNE

if pairs_path:
    ds_train = tf.data.Dataset.from_tensor_slices((X1[train_idx], X2[train_idx]))
    ds_val   = tf.data.Dataset.from_tensor_slices((X1[val_idx], X2[val_idx]))
else:
    ds_train = tf.data.Dataset.from_tensor_slices(X[train_idx])
    ds_val   = tf.data.Dataset.from_tensor_slices(X[val_idx])

    ds_train = ds_train.map(lambda x: make_pair_from_seq(x), num_parallel_calls=AUTOTUNE)
    ds_val   = ds_val.map(lambda x: make_pair_from_seq(x), num_parallel_calls=AUTOTUNE)

ds_train = ds_train.shuffle(4096, seed=SEED, reshuffle_each_iteration=True).batch(BATCH).prefetch(AUTOTUNE)
ds_val   = ds_val.batch(BATCH).prefetch(AUTOTUNE)

next(iter(ds_train))[0].shape, next(iter(ds_train))[1].shape


(TensorShape([4, 128, 3]), TensorShape([4, 128, 3]))

In [9]:
# -----------------------
# Encoder + Projection head (CNN + GRU)
# -----------------------
def build_encoder(timesteps: int, n_features: int, emb_dim: int = 128):
    inp = keras.Input(shape=(timesteps, n_features))

    x = layers.Conv1D(64, 5, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)

    x = layers.GRU(128, return_sequences=False)(x)
    x = layers.Dense(emb_dim)(x)
    x = layers.LayerNormalization()(x)
    return keras.Model(inp, x, name="encoder")

def build_projector(emb_dim: int = 128, proj_dim: int = 128):
    inp = keras.Input(shape=(emb_dim,))
    x = layers.Dense(256, activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(proj_dim)(x)
    return keras.Model(inp, x, name="projector")

ENC_DIM = 128
PROJ_DIM = 128

encoder = build_encoder(T, F, emb_dim=ENC_DIM)
projector = build_projector(ENC_DIM, PROJ_DIM)

encoder.summary()


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 128, 64)        │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, 128)            │           256 │
│ (LayerNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,720 (620.00 KB)

 Trainable params: 158,336 (618.50 KB)

 Non-trainable params: 384 (1.50 KB)

In [10]:
# -----------------------
# NT-Xent loss (SimCLR)
# -----------------------
def l2_normalize(x, axis=1, eps=1e-9):
    return x / (tf.norm(x, axis=axis, keepdims=True) + eps)

@tf.function
def nt_xent_loss(z1, z2, temperature=0.2):
    # z1, z2: (B, D)
    z1 = l2_normalize(z1, axis=1)
    z2 = l2_normalize(z2, axis=1)

    B = tf.shape(z1)[0]
    z = tf.concat([z1, z2], axis=0)  # (2B, D)

    # cosine similarity matrix
    sim = tf.matmul(z, z, transpose_b=True)  # (2B, 2B)
    sim = sim / temperature

    # mask self-similarity
    large_neg = tf.cast(1e9, sim.dtype)
    sim = sim - tf.eye(2*B, dtype=sim.dtype) * large_neg

    # positives: i <-> i+B
    pos = tf.concat([tf.range(B, 2*B), tf.range(0, B)], axis=0)  # (2B,)
    labels = pos

    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=sim)
    return tf.reduce_mean(loss)



In [11]:
import tensorflow as tf
from tensorflow import keras
import os

# -----------------------
# Improved SimCLR Model
# -----------------------
class SimCLR(keras.Model):
    def __init__(self, encoder, projector, temperature=0.5):
        super().__init__()
        self.encoder = encoder
        self.projector = projector
        self.temperature = temperature
        
        # Track metrics
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.val_loss_tracker = keras.metrics.Mean(name="val_loss")

    def call(self, inputs, training=False):
        x1, x2 = inputs
        h1 = self.encoder(x1, training=training)
        h2 = self.encoder(x2, training=training)
        z1 = self.projector(h1, training=training)
        z2 = self.projector(h2, training=training)
        
        # L2 normalize embeddings
        z1 = tf.math.l2_normalize(z1, axis=1)
        z2 = tf.math.l2_normalize(z2, axis=1)
        
        return z1, z2

    def train_step(self, data):
        x1, x2 = data
        
        with tf.GradientTape() as tape:
            z1, z2 = self((x1, x2), training=True)
            loss = nt_xent_loss(z1, z2, temperature=self.temperature)
            
            # Add small regularization to prevent collapse
            loss = loss + 1e-6

        # Compute gradients
        grads = tape.gradient(loss, self.trainable_variables)
        
        # Clip gradients to prevent explosion
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        
        # Apply gradients
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        
        # Update metrics
        self.loss_tracker.update_state(loss)
        
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        x1, x2 = data
        z1, z2 = self((x1, x2), training=False)
        loss = nt_xent_loss(z1, z2, temperature=self.temperature)
        
        # Update validation metrics
        self.val_loss_tracker.update_state(loss)
        
        return {"loss": self.val_loss_tracker.result()}
    
    @property
    def metrics(self):
        return [self.loss_tracker, self.val_loss_tracker]


# -----------------------
# Improved NT-Xent Loss
# -----------------------
def nt_xent_loss(z1, z2, temperature=0.5):
    """
    Normalized Temperature-scaled Cross Entropy Loss
    """
    batch_size = tf.shape(z1)[0]
    
    # Normalize embeddings (if not done in call)
    z1 = tf.math.l2_normalize(z1, axis=1)
    z2 = tf.math.l2_normalize(z2, axis=1)
    
    # Concatenate embeddings
    z = tf.concat([z1, z2], axis=0)  # Shape: (2*batch_size, embedding_dim)
    
    # Compute cosine similarity matrix
    similarity_matrix = tf.matmul(z, z, transpose_b=True)  # (2B, 2B)
    similarity_matrix = similarity_matrix / temperature
    
    # Create mask to exclude self-similarity
    mask = tf.eye(2 * batch_size, dtype=tf.bool)
    similarity_matrix = tf.where(mask, -1e9, similarity_matrix)
    
    # Positive pairs: (i, i+batch_size) and (i+batch_size, i)
    positives = tf.concat([
        tf.linalg.diag_part(similarity_matrix[:batch_size, batch_size:]),
        tf.linalg.diag_part(similarity_matrix[batch_size:, :batch_size])
    ], axis=0)
    
    # Compute log softmax
    negatives = tf.reduce_logsumexp(similarity_matrix, axis=1)
    
    # Contrastive loss
    loss = -positives + negatives
    loss = tf.reduce_mean(loss)
    
    return loss


# -----------------------
# Training Setup
# -----------------------
# Calculate steps
steps_per_epoch = len(ds_train)
total_steps = steps_per_epoch * 100

# Simple learning rate schedule - CosineDecay without warmup
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=total_steps,
    alpha=0.1  # minimum learning rate = 0.1 * initial_lr
)

# Initialize model with better temperature
model = SimCLR(encoder, projector, temperature=0.5)

# Use Adam optimizer
optimizer = keras.optimizers.Adam(
    learning_rate=lr_schedule,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7
)

model.compile(optimizer=optimizer)

# Build model
x1_batch, x2_batch = next(iter(ds_train))
_ = model((x1_batch, x2_batch), training=False)

# -----------------------
# Improved Callbacks
# -----------------------
callbacks = [
    # Model checkpoint
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CKPT_DIR, "simclr_best.weights.h5"),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    
    # Early stopping with more patience
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=20,  # More patience for contrastive learning
        restore_best_weights=True,
        verbose=1,
        min_delta=1e-4,  # Minimum change to qualify as improvement
    ),
    
    # Reduce LR on plateau (backup if cosine decay isn't enough)
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
    
    # TensorBoard logging
    keras.callbacks.TensorBoard(
        log_dir=os.path.join(CKPT_DIR, "logs"),
        histogram_freq=1,
        write_graph=True,
    ),
    
    # Log learning rate
    keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: logs.update({
            'lr': float(model.optimizer.learning_rate(model.optimizer.iterations))
        })
    ),
]

# -----------------------
# Training
# -----------------------
print("Starting training...")
print(f"Temperature: {model.temperature}")
print(f"Initial LR: {1e-3}")
print(f"Batch size per view: {x1_batch.shape[0]}")

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=callbacks,
    verbose=1,
)

# -----------------------
# Save final model
# -----------------------
print("\nSaving final models...")
encoder.save(os.path.join(CKPT_DIR, "encoder_final.keras"))
model.save_weights(os.path.join(CKPT_DIR, "simclr_final.weights.h5"))

print("\n✅ Training complete!")
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")

Starting training...
Temperature: 0.5
Initial LR: 0.001
Batch size per view: 4
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - loss: 1.5819
Epoch 1: val_loss improved from None to 0.00000, saving model to checkpoints\simclr_best.weights.h5

Epoch 1: finished saving model to checkpoints\simclr_best.weights.h5


TypeError: 'tensorflow.python.framework.ops.EagerTensor' object is not callable

In [ ]:
# -----------------------
# Extract embeddings (encoder output) for validation set
# -----------------------
def get_embeddings(ds, encoder):
    outs = []
    for x1, x2 in ds:
        h = encoder(x1, training=False)  # use only view1 for evaluation
        outs.append(h.numpy())
    return np.concatenate(outs, axis=0)

val_emb = get_embeddings(ds_val, encoder)
print("val_emb:", val_emb.shape)

# Unsupervised clustering quality (proxy)
k = 10
km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
cl = km.fit_predict(val_emb)
sil = silhouette_score(val_emb, cl)
print("Silhouette (k=10):", sil)


In [ ]:
# Save embeddings for later analysis / plotting notebooks
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)
np.savez_compressed(
    os.path.join(out_dir, "val_embeddings_simclr.npz"),
    embeddings=val_emb,
    cluster_labels=cl,
    oid=(oid[val_idx] if oid is not None else None),
)
print("Saved:", os.path.join(out_dir, "val_embeddings_simclr.npz"))
